# Block GAPHC Kumaraswamy estimation — Mathematical formulation

This notebook implements and demonstrates the model, censoring scheme, maximum
likelihood estimation, and Bayesian estimation of kumaraswamy distribution.

**What this notebook covers**
1. The block Kumaraswamy model and the block GAPHC censoring scheme (Section 2.1–2.2)
2. The likelihood and unrestricted MLE (Section 2.3–2.4)
3. Order-restricted MLE via PAVA (Section 2.5)
4. Fisher-information confidence intervals (Section 2.6)
5. Pivotal-based exact confidence intervals for $\alpha$, per block (Section 2.7, working assumption from Wu, Tseng & Kus 2025)
6. Bayesian estimation, unrestricted and order-restricted (Section 3)

Section 4 (simulation study and real-data application) is demonstrated separately
in the write-up; this notebook focuses on showing the estimators work correctly on
a small worked example, with every number checked against the paper's formulas.

## 1. Setup

We assume $k$ independent blocks, each Kumaraswamy with a common $\alpha$ and
block-specific $\lambda_i$:
$$f_i(x;\alpha,\lambda_i)=\alpha\lambda_i x^{\alpha-1}(1-x^\alpha)^{\lambda_i-1},\qquad
S_i(x;\alpha,\lambda_i)=(1-x^\alpha)^{\lambda_i},\quad 0<x<1.$$
Recall (Remark 1 in the paper): larger $\lambda_i$ means *shorter* lifetimes, so the
order restriction $\lambda_1\le\cdots\le\lambda_k$ means block 1 has the
stochastically longest lifetimes.

In [ ]:
import numpy as np
from scipy.optimize import minimize_scalar, brentq
from scipy.stats import chi2, f as fdist, gamma as gammadist

rng_global = np.random.default_rng(0)

def rkum(n, alpha, lam, rng):
    """Draw n i.i.d. Kumaraswamy(alpha, lam) values via inverse-transform sampling."""
    u = rng.uniform(size=n)
    return (1 - (1 - u) ** (1 / lam)) ** (1 / alpha)

## 2. Block generalized adaptive progressive hybrid censoring (Section 2.2)

In block $i$, $n_i$ units start the test with a target number of failures $r_i$ and
two thresholds $T_{1i}<T_{2i}$. The realized scheme follows one of three cases:

- **Case I** ($X_{i,r_i}<T_{1i}$): stop at the $r_i$-th failure, planned removals used as is.
- **Case II** ($T_{1i}<X_{i,r_i}<T_{2i}$): no further removals after $T_{1i}$; the
  test continues to the $r_i$-th failure, where all remaining units are removed.
- **Case III** ($X_{i,r_i}>T_{2i}$): the test is censored at $T_{2i}$ with $d_2$
  observed failures; the surviving units are censored there ($C_i>0$).

`gen_block` below generates one block's data under this rule, using the
Balakrishnan–Sandhu algorithm for the progressive-censoring order statistics.

In [ ]:
def _rprogressive(n, m, R, alpha, lam, rng):
    """Progressively Type-II censored sample of size m from n units, scheme R."""
    R = np.asarray(R)
    W = rng.uniform(size=m)
    i = np.arange(1, m + 1)
    tail = np.array([R[m - k:].sum() for k in i])
    V = W ** (1 / (i + tail))
    U = 1 - np.cumprod(V[::-1])
    return np.sort((1 - (1 - U) ** (1 / lam)) ** (1 / alpha))

def gen_block(n, r_target, T1, T2, alpha, lam, rng):
    """Generate one block under the block GAPHC scheme (Section 2.2, Cases I-III)."""
    R_plan = [0]*(r_target-1) + [n - r_target]      # Type-II-style planned scheme
    x = _rprogressive(n, r_target, R_plan, alpha, lam, rng)
    if x[-1] < T1:
        return dict(x=list(x), R=R_plan, tau=float(x[-1]), C=0, m=r_target, case=1)
    d1 = int(np.searchsorted(x, T1, side='right'))
    if x[-1] < T2:
        R_adapt = [0]*(r_target-1) + [n - r_target]  # leftover dumped at final failure
        return dict(x=list(x), R=R_adapt, tau=float(x[-1]), C=0, m=r_target, case=2)
    # Case III: regenerate a complete sample and censor directly at T2
    full = np.sort(rkum(n, alpha, lam, rng))
    d2 = max(int(np.searchsorted(full, T2, side='right')), 1)
    xobs = full[:d2]
    return dict(x=list(xobs), R=[0]*d2, tau=float(T2), C=int(n-d2), m=int(d2), case=3)

### Demonstration: generate a small 3-block dataset

We'll use this one dataset throughout the notebook to show every estimator.

In [ ]:
rng = np.random.default_rng(42)

TRUE_ALPHA = 2.0
TRUE_LAMBDA = [1.5, 2.5, 4.0]   # increasing -> satisfies the order restriction
n_list  = [40, 45, 50]
r_list  = [28, 32, 35]
T1_list = [0.30, 0.28, 0.25]
T2_list = [0.65, 0.60, 0.55]

blocks = []
for i in range(3):
    b = gen_block(n_list[i], r_list[i], T1_list[i], T2_list[i],
                  TRUE_ALPHA, TRUE_LAMBDA[i], rng)
    blocks.append(b)
    print(f"Block {i+1}: n={n_list[i]}, target r={r_list[i]} -> "
          f"case {b['case']}, m={b['m']} observed, C={b['C']} censored at tau={b['tau']:.3f}")

## 3. Likelihood and unrestricted MLE (Section 2.3–2.4)

From Eq. (5)-(7) of the paper, define
$$W_i(\alpha) = -\Big[\sum_{j=1}^{m_i}(1+R_{ij})\log(1-x_{ij}^\alpha) + C_i\log(1-\tau_i^\alpha)\Big].$$
For fixed $\alpha$, the conditional MLE of $\lambda_i$ is $\hat\lambda_i(\alpha)=m_i/W_i(\alpha)$
(Eq. 10), and $\alpha$ is found by maximizing the profile log-likelihood (Eq. 12-13).

**Note on the sign of $W_i(\alpha)$:** this is the term that was flagged earlier as a
bug in an older version of this code (the sign on $\sum\log(1-x_{ij}^\alpha)$ in the
log-likelihood, Eq. 8). The implementation below matches the paper's Eq. (7)-(8).

In [ ]:
def Wi(alpha, x, R, C, tau):
    """Eq. (7): block i's W_i(alpha)."""
    x = np.asarray(x); R = np.asarray(R)
    s = np.sum((1 + R) * np.log(1 - x**alpha))
    if C > 0:
        s += C * np.log(1 - tau**alpha)
    return -s

def neg_loglik_block(alpha, blocks):
    """Negative profile log-likelihood, Eq. (8) with lambda_i profiled out via Eq. (10)."""
    if alpha <= 0:
        return np.inf
    tot = 0.0
    for b in blocks:
        W = Wi(alpha, b['x'], b['R'], b['C'], b['tau'])
        if W <= 0 or b['m'] == 0:
            return np.inf
        lam_hat = b['m'] / W
        x = np.asarray(b['x'])
        ll = (b['m']*np.log(alpha) + b['m']*np.log(lam_hat)
              + (alpha-1)*np.sum(np.log(x)) - np.sum(np.log(1-x**alpha))
              - lam_hat*W)
        tot += ll
    return -tot

def fit_unrestricted(blocks, bounds=(0.05, 30)):
    """Eq. (13)-(14): unrestricted MLE of alpha and each lambda_i."""
    res = minimize_scalar(neg_loglik_block, bounds=bounds, args=(blocks,), method='bounded')
    alpha = res.x
    lams = [b['m'] / Wi(alpha, b['x'], b['R'], b['C'], b['tau']) for b in blocks]
    return alpha, np.array(lams)

In [ ]:
alpha_hat, lambda_hat = fit_unrestricted(blocks)
print(f"True alpha = {TRUE_ALPHA}, True lambda = {TRUE_LAMBDA}")
print(f"Unrestricted MLE: alpha_hat = {alpha_hat:.4f}")
print(f"Unrestricted MLE: lambda_hat = {np.round(lambda_hat, 4)}")
print(f"\nOrdering lambda_1 <= lambda_2 <= lambda_3 holds: {np.all(np.diff(lambda_hat) >= 0)}")

## 4. Order-restricted MLE via PAVA (Section 2.5)

When $\hat\lambda_i(\alpha)$ violates $\lambda_1\le\cdots\le\lambda_k$, adjacent
blocks are pooled (Eq. 20):
$$\hat\lambda_{r:s}(\alpha) = \frac{\sum_{i=r}^s m_i}{\sum_{i=r}^s W_i(\alpha)}.$$
This is exactly the pool-adjacent-violators algorithm (PAVA), weighted by
$W_i(\alpha)$. In our demonstration dataset the unrestricted estimates already
respect the order (since the true $\lambda_i$ are well separated), so we also show
a second, deliberately *unordered* dataset to see the pooling actually trigger.

In [ ]:
def restricted_lambda(blocks, alpha):
    """PAVA pooling of the conditional lambda_i(alpha) estimates (Eq. 16-20)."""
    m = np.array([b['m'] for b in blocks], dtype=float)
    W = np.array([Wi(alpha, b['x'], b['R'], b['C'], b['tau']) for b in blocks])
    k = len(m)
    groups = [[m[i], W[i], 1] for i in range(k)]   # [sum_m, sum_W, block_count]
    changed = True
    while changed:
        changed = False
        for idx in range(len(groups) - 1):
            v1 = groups[idx][0] / groups[idx][1]
            v2 = groups[idx + 1][0] / groups[idx + 1][1]
            if v1 > v2:
                merged = [groups[idx][0] + groups[idx+1][0],
                          groups[idx][1] + groups[idx+1][1],
                          groups[idx][2] + groups[idx+1][2]]
                groups = groups[:idx] + [merged] + groups[idx+2:]
                changed = True
                break
    lam_r = []
    for sm, sw, cnt in groups:
        lam_r += [sm / sw] * cnt
    return np.array(lam_r)

def neg_loglik_restricted(alpha, blocks):
    """Eq. (21): restricted profile log-likelihood."""
    if alpha <= 0:
        return np.inf
    lam_r = restricted_lambda(blocks, alpha)
    tot = 0.0
    for b, lam in zip(blocks, lam_r):
        W = Wi(alpha, b['x'], b['R'], b['C'], b['tau'])
        if W <= 0 or b['m'] == 0:
            return np.inf
        x = np.asarray(b['x'])
        ll = (b['m']*np.log(alpha) + b['m']*np.log(lam)
              + (alpha-1)*np.sum(np.log(x)) - np.sum(np.log(1-x**alpha))
              - lam*W)
        tot += ll
    return -tot

def fit_restricted(blocks, bounds=(0.05, 30)):
    """Eq. (22)-(23): order-restricted MLE."""
    res = minimize_scalar(neg_loglik_restricted, bounds=bounds, args=(blocks,), method='bounded')
    alpha = res.x
    return alpha, restricted_lambda(blocks, alpha)

In [ ]:
alpha_r, lambda_r = fit_restricted(blocks)
print("Our demonstration dataset (already ordered):")
print(f"  Restricted MLE: alpha = {alpha_r:.4f}, lambda = {np.round(lambda_r, 4)}")
print(f"  (matches unrestricted since no pooling was needed: "
      f"{np.allclose(lambda_r, lambda_hat, atol=1e-3)})")

# Now a second dataset where the true lambdas are close, to force pooling
rng2 = np.random.default_rng(7)
blocks_close = []
TRUE_LAMBDA_CLOSE = [2.5, 2.6, 2.4]   # not monotone -> likely to need pooling
for i in range(3):
    b = gen_block(n_list[i], r_list[i], T1_list[i], T2_list[i],
                  TRUE_ALPHA, TRUE_LAMBDA_CLOSE[i], rng2)
    blocks_close.append(b)

a_u2, l_u2 = fit_unrestricted(blocks_close)
a_r2, l_r2 = fit_restricted(blocks_close)
print("\nSecond dataset (true lambda not monotone, to trigger pooling):")
print(f"  Unrestricted: alpha={a_u2:.4f}, lambda={np.round(l_u2, 4)}  "
      f"(ordered: {np.all(np.diff(l_u2) >= -1e-9)})")
print(f"  Restricted:   alpha={a_r2:.4f}, lambda={np.round(l_r2, 4)}  "
      f"(ordered: {np.all(np.diff(l_r2) >= -1e-9)})")

## 5. Fisher-information confidence intervals (Section 2.6)

$J(\theta)=-\partial^2\ell/\partial\theta\partial\theta^\top$, evaluated at the
unrestricted MLE, gives approximate standard errors via
$\widehat{\mathrm{Cov}}(\hat\theta)=J(\hat\theta)^{-1}$. We compute $J$ numerically
here (finite differences) rather than from the analytic second derivatives, since
the paper's formulas are block-additive and a numerical Hessian is simpler to keep
correct alongside the rest of this notebook.

In [ ]:
def neg_loglik_full(theta, blocks):
    """Full (non-profiled) negative log-likelihood, theta = [alpha, lambda_1,...,lambda_k]."""
    alpha = theta[0]; lams = theta[1:]
    if alpha <= 0 or np.any(lams <= 0):
        return np.inf
    tot = 0.0
    for b, lam in zip(blocks, lams):
        x = np.asarray(b['x']); R = np.asarray(b['R'])
        W = Wi(alpha, b['x'], b['R'], b['C'], b['tau'])
        ll = (b['m']*np.log(alpha) + b['m']*np.log(lam)
              + (alpha-1)*np.sum(np.log(x)) - np.sum(np.log(1-x**alpha))
              - lam*W)
        tot += ll
    return -tot

def numeric_hessian(f, x0, h=1e-4):
    n = len(x0)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            xp = x0.copy(); xp[i]+=h; xp[j]+=h
            xpm = x0.copy(); xpm[i]+=h; xpm[j]-=h
            xmp = x0.copy(); xmp[i]-=h; xmp[j]+=h
            xm = x0.copy(); xm[i]-=h; xm[j]-=h
            H[i,j] = (f(xp)-f(xpm)-f(xmp)+f(xm))/(4*h*h)
    return H

theta_hat = np.r_[alpha_hat, lambda_hat]
J = numeric_hessian(lambda t: neg_loglik_full(t, blocks), theta_hat)   # already -loglik, so J = Hessian of -ll
cov = np.linalg.inv(J)
se = np.sqrt(np.diag(cov))
z = 1.96
print("95% Fisher-information CIs:")
names = ['alpha'] + [f'lambda_{i+1}' for i in range(3)]
for name, est, s in zip(names, theta_hat, se):
    print(f"  {name}: {est:.4f} +/- {z*s:.4f}  ->  ({est - z*s:.4f}, {est + z*s:.4f})")

## 6. Pivotal-based exact confidence interval for $\alpha$, per block (Section 2.7)

As a working assumption from Wu, Tseng & Kus (2025), within a block in Cases I/II we
can treat the censoring as a fixed progressive scheme and use their Theorem 2. Define
$$T_i(\alpha)=\frac{1}{n_i(m_i-1)}\sum_{j=1}^{m_i}(1+R_{ij})\frac{\log(1-x_{ij}^\alpha)}{\log(1-x_{i1}^\alpha)}-\frac{1}{m_i-1},$$
which has an $F_{2m_i-2,\,2}$ distribution and is monotone in $\alpha$. Inverting it
at the two $F$-quantiles gives an exact interval for $\alpha$ using only that
block's data.

In [ ]:
def alpha_pivot_ci(block, n_i, gamma=0.05, lo_b=0.05, hi_b=15):
    """Exact 100(1-gamma)% CI for alpha from one block (Wu, Tseng & Kus 2025, Thm 2)."""
    x = np.asarray(block['x']); R = np.asarray(block['R']); m = block['m']
    def T(a):
        num = np.sum((1+R) * np.log(1 - x**a) / np.log(1 - x[0]**a))
        return num / (n_i*(m-1)) - 1/(m-1)
    t_lo = fdist.ppf(1-gamma/2, 2*m-2, 2)
    t_hi = fdist.ppf(gamma/2, 2*m-2, 2)
    def solve(t):
        f = lambda a: T(a) - t
        grid = np.linspace(lo_b, hi_b, 400)
        vals = [f(g) for g in grid]
        for i in range(len(grid)-1):
            if np.isfinite(vals[i]) and np.isfinite(vals[i+1]) and vals[i]*vals[i+1] < 0:
                return brentq(f, grid[i], grid[i+1])
        return np.nan
    a, b_ = solve(t_lo), solve(t_hi)
    return (min(a, b_), max(a, b_))

print("Per-block exact 95% CI for alpha (true alpha = 2.0):")
for i, (b, n_i) in enumerate(zip(blocks, n_list)):
    if b['case'] == 3:
        print(f"  Block {i+1}: Case III -- pivot not derived for this case, skipped")
        continue
    lo, hi = alpha_pivot_ci(b, n_i)
    print(f"  Block {i+1} (case {b['case']}, m={b['m']}): ({lo:.3f}, {hi:.3f})")

## 7. Bayesian estimation (Section 3)

Independent $\mathrm{Gamma}(a_0,b_0)$ and $\mathrm{Gamma}(a_i,b_i)$ priors (shape-rate)
for $\alpha$ and each $\lambda_i$. The full conditional of $\lambda_i\mid\alpha$ is
exactly $\mathrm{Gamma}(a_i+m_i,\,b_i+W_i(\alpha))$ (Eq. 38 structure), sampled via
Gibbs; $\alpha$ has no closed-form conditional and is updated with a
random-walk Metropolis step on $\log\alpha$. Under the order restriction, each
$\lambda_i\mid\alpha,\lambda_{-i}$ is the same Gamma distribution truncated to
$[\lambda_{i-1},\lambda_{i+1}]$ (Section 3.4).

In [ ]:
def logpost_alpha(a, lam, blocks, a0, b0):
    if a <= 0:
        return -np.inf
    lp = (a0-1)*np.log(a) - b0*a
    for b, l in zip(blocks, lam):
        x = np.asarray(b['x'])
        W = Wi(a, b['x'], b['R'], b['C'], b['tau'])
        lp += b['m']*np.log(a) + (a-1)*np.sum(np.log(x)) - np.sum(np.log(1-x**a)) - l*W
    return lp

def gibbs_unrestricted(blocks, a0, b0, ai, bi, n_iter=4000, burn=1000, seed=0):
    rng = np.random.default_rng(seed)
    k = len(blocks)
    alpha = fit_unrestricted(blocks)[0]
    lam = np.array([b['m']/Wi(alpha, b['x'], b['R'], b['C'], b['tau']) for b in blocks])
    draws_a = np.empty(n_iter); draws_l = np.empty((n_iter, k))
    step = 0.15
    cur_lp = logpost_alpha(alpha, lam, blocks, a0, b0)
    for it in range(n_iter):
        for idx, b in enumerate(blocks):
            W = Wi(alpha, b['x'], b['R'], b['C'], b['tau'])
            lam[idx] = rng.gamma(ai[idx]+b['m'], 1/(bi[idx]+W))
        prop = alpha * np.exp(step*rng.normal())
        prop_lp = logpost_alpha(prop, lam, blocks, a0, b0)
        if np.log(rng.uniform()) < prop_lp - cur_lp:
            alpha = prop; cur_lp = prop_lp
        draws_a[it] = alpha; draws_l[it] = lam
    return draws_a[burn:], draws_l[burn:]

def trunc_gamma(shape, rate, lo, hi, rng):
    lo_c = 0.0 if lo <= 0 else gammadist.cdf(lo, a=shape, scale=1/rate)
    hi_c = 1.0 if hi == np.inf else gammadist.cdf(hi, a=shape, scale=1/rate)
    if hi_c <= lo_c:
        hi_c = lo_c + 1e-9
    u = rng.uniform(lo_c, hi_c)
    return gammadist.ppf(u, a=shape, scale=1/rate)

def gibbs_restricted(blocks, a0, b0, ai, bi, n_iter=4000, burn=1000, seed=1):
    rng = np.random.default_rng(seed)
    k = len(blocks)
    alpha = fit_restricted(blocks)[0]
    lam = restricted_lambda(blocks, alpha).copy()
    draws_a = np.empty(n_iter); draws_l = np.empty((n_iter, k))
    step = 0.15
    cur_lp = logpost_alpha(alpha, lam, blocks, a0, b0)
    for it in range(n_iter):
        for idx, b in enumerate(blocks):
            W = Wi(alpha, b['x'], b['R'], b['C'], b['tau'])
            lo = lam[idx-1] if idx > 0 else 0.0
            hi = lam[idx+1] if idx < k-1 else np.inf
            lam[idx] = trunc_gamma(ai[idx]+b['m'], bi[idx]+W, lo, hi, rng)
        prop = alpha * np.exp(step*rng.normal())
        prop_lp = logpost_alpha(prop, lam, blocks, a0, b0)
        if np.log(rng.uniform()) < prop_lp - cur_lp:
            alpha = prop; cur_lp = prop_lp
        draws_a[it] = alpha; draws_l[it] = lam.copy()
    return draws_a[burn:], draws_l[burn:]

def hpd(draws, cred=0.95):
    d = np.sort(draws); n = len(d); h = int(np.floor(cred*n))
    widths = d[h:] - d[:n-h]
    j = np.argmin(widths)
    return d[j], d[j+h]

In [ ]:
a0, b0 = 2.0, 1.0
ai = [2.0, 2.0, 2.0]; bi = [1.0, 1.0, 1.0]

draws_a_u, draws_l_u = gibbs_unrestricted(blocks, a0, b0, ai, bi, n_iter=4000, burn=1000, seed=11)
draws_a_r, draws_l_r = gibbs_restricted(blocks, a0, b0, ai, bi, n_iter=4000, burn=1000, seed=22)

print(f"True alpha = {TRUE_ALPHA}, True lambda = {TRUE_LAMBDA}\n")

print("Unrestricted Bayes (posterior mean, squared-error loss):")
print(f"  alpha: {draws_a_u.mean():.4f}, 95% HPD {hpd(draws_a_u)}")
for i in range(3):
    print(f"  lambda_{i+1}: {draws_l_u[:,i].mean():.4f}, 95% HPD {hpd(draws_l_u[:,i])}")

print("\nRestricted Bayes (posterior mean, squared-error loss):")
print(f"  alpha: {draws_a_r.mean():.4f}, 95% HPD {hpd(draws_a_r)}")
for i in range(3):
    print(f"  lambda_{i+1}: {draws_l_r.mean(axis=0)[i]:.4f}, 95% HPD {hpd(draws_l_r[:,i])}")
print(f"\nRestricted posterior means respect the order: "
      f"{np.all(np.diff(draws_l_r.mean(axis=0)) >= 0)}")

## Summary

This notebook reproduced, on one worked dataset:
- the block GAPHC data generator and its three censoring cases (Section 2.2)
- the unrestricted MLE via profile likelihood (Section 2.3–2.4)
- the order-restricted MLE via PAVA pooling, and a second dataset where pooling
  actually triggers (Section 2.5)
- Fisher-information confidence intervals (Section 2.6)
- the per-block exact pivotal confidence interval for $\alpha$ (Section 2.7)
- unrestricted and order-restricted Bayesian estimation via Gibbs/Metropolis–Hastings
  (Section 3)

The full simulation study and the real insulating-fluid data analysis (Section 4)
use the same functions at larger scale and are reported in the LaTeX write-up
rather than reproduced here.